In [36]:
import os
import json
import numpy as np
import pandas as pd
from obspy import read
from datetime import datetime, timedelta, timezone

# KONFIGURASI
WAVEFORM_ROOT = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02'
CATALOG_CSV = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog_CLEAN.csv'
OUTPUT_JSON = "/Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json"

def validate_catalog_vs_waveform(catalog_csv, waveform_root):
    print("[INFO] Membaca Katalog...")
    df = pd.read_csv(catalog_csv)
    df['origin_dt'] = pd.to_datetime(df['origin_time'], utc=True)
    
    matches = []
    
    print(f"[INFO] Memindai file di {waveform_root}...")
    for root, _, files in os.walk(waveform_root):
        for fn in files:
            if not fn.lower().endswith(".mseed"): continue
            
            # Ekstraksi ID dari nama file
            if "BMKG-" not in fn: continue
            start = fn.find("BMKG-")
            event_id = fn[start:].replace('.mseed', '')
            
            # Cek di Katalog
            match = df[df['Event ID'] == event_id]
            if match.empty: continue
            
            # Verifikasi Waveform
            try:
                st = read(os.path.join(root, fn))
                tr = st[0]
                origin = match.iloc[0]['origin_dt']
                
                # Cek apakah origin time ada dalam rentang waveform
                if tr.stats.starttime <= origin <= tr.stats.endtime:
                    matches.append((event_id, "VALID - Inside Waveform"))
                else:
                    matches.append((event_id, f"WARNING - Origin outside: {origin} vs Range: {tr.stats.starttime} to {tr.stats.endtime}"))
            except Exception as e:
                matches.append((event_id, f"ERROR - {str(e)}"))
                
    # Tampilkan hasil validasi
    print("\n--- HASIL VALIDASI ---")
    for event_id, status in matches[:20]: # Tampilkan 20 contoh
        print(f"ID: {event_id} | Status: {status}")
    
    print(f"\nTotal Event yang ditemukan: {len(matches)}")
    return matches

validate_catalog_vs_waveform(CATALOG_CSV, WAVEFORM_ROOT)

[INFO] Membaca Katalog...
[INFO] Memindai file di /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02...

--- HASIL VALIDASI ---
ID: BMKG-20130608084005-001 | Status: VALID - Inside Waveform
ID: BMKG-20130219204418-001 | Status: VALID - Inside Waveform
ID: BMKG-20130222001303-001 | Status: VALID - Inside Waveform
ID: BMKG-20130602200319-001 | Status: VALID - Inside Waveform
ID: BMKG-20130201224555-001 | Status: VALID - Inside Waveform
ID: BMKG-20130325145629-001 | Status: VALID - Inside Waveform
ID: BMKG-20130310084302-001 | Status: VALID - Inside Waveform
ID: BMKG-20130223230717-001 | Status: VALID - Inside Waveform
ID: BMKG-20130313180805-001 | Status: VALID - Inside Waveform
ID: BMKG-20131022110200-001 | Status: VALID - Inside Waveform
ID: BMKG-20130709145324-001 | Status: VALID - Inside Waveform
ID: BMKG-20130313212313-001 | Status: VALID - Inside Waveform
ID: BMKG-20131023183033-001 | Stat

[('BMKG-20130608084005-001', 'VALID - Inside Waveform'),
 ('BMKG-20130219204418-001', 'VALID - Inside Waveform'),
 ('BMKG-20130222001303-001', 'VALID - Inside Waveform'),
 ('BMKG-20130602200319-001', 'VALID - Inside Waveform'),
 ('BMKG-20130201224555-001', 'VALID - Inside Waveform'),
 ('BMKG-20130325145629-001', 'VALID - Inside Waveform'),
 ('BMKG-20130310084302-001', 'VALID - Inside Waveform'),
 ('BMKG-20130223230717-001', 'VALID - Inside Waveform'),
 ('BMKG-20130313180805-001', 'VALID - Inside Waveform'),
 ('BMKG-20131022110200-001', 'VALID - Inside Waveform'),
 ('BMKG-20130709145324-001', 'VALID - Inside Waveform'),
 ('BMKG-20130313212313-001', 'VALID - Inside Waveform'),
 ('BMKG-20131023183033-001', 'VALID - Inside Waveform'),
 ('BMKG-20130918092656-001', 'VALID - Inside Waveform'),
 ('BMKG-20130326145621-001', 'VALID - Inside Waveform'),
 ('BMKG-20130614010456-001', 'VALID - Inside Waveform'),
 ('BMKG-20130203113712-001', 'VALID - Inside Waveform'),
 ('BMKG-20130209021937-001', 'V

In [49]:
import json
import pandas as pd
from pathlib import Path
from obspy import read
from datetime import timedelta

# KONFIGURASI
WAVEFORM_ROOT = Path('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready')
CATALOG_CSV = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog_CLEAN.csv'
OUTPUT_JSON = "/Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json"

def build_json_final():
    print("[INFO] Loading Katalog...")
    df = pd.read_csv(CATALOG_CSV)
    df['origin_dt'] = pd.to_datetime(df['origin_time'], utc=True)
    
    output = {}
    count = 0
    
    # Menggunakan rglob untuk memindai semua sub-folder secara rekursif
    all_files = list(WAVEFORM_ROOT.rglob("*.mseed"))
    print(f"[INFO] Memulai ekstraksi dari {len(all_files)} file...")
    
    for file_path in all_files:
        try:
            # 1. Baca metadata file untuk mendapatkan waktu awal (headonly lebih cepat)
            st = read(str(file_path), headonly=True)
            file_start = pd.to_datetime(st[0].stats.starttime.datetime, utc=True)
            
            # 2. Cari gempa di katalog yang paling dekat (selisih < 60 detik)
            df['diff'] = (df['origin_dt'] - file_start).abs()
            best_match_idx = df['diff'].idxmin()
            best_match = df.iloc[best_match_idx]
            
            if best_match['diff'] < timedelta(seconds=60):
                # 3. Baca data penuh untuk ekstraksi identik
                st_full = read(str(file_path))
                tr = st_full[0]
                
                # Gunakan Event ID sebagai kunci (sesuai format Zhi Geng)
                event_id = str(best_match['Event ID'])
                
                output[event_id] = {
                    "type": "se",
                    "Z": extract_window(tr, best_match['origin_dt']),
                    "Z_noise": extract_noise_window(tr, best_match['origin_dt'])
                }
                
                count += 1
                if count % 500 == 0: 
                    print(f"Berhasil memproses: {count} event...")
        
        except Exception:
            continue

    # Menyimpan file JSON
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n[SUKSES] Total {count} event tersimpan di {OUTPUT_JSON}")

if __name__ == "__main__":
    build_json_final()

[INFO] Loading Katalog...
[INFO] Memulai ekstraksi dari 24183 file...

[SUKSES] Total 0 event tersimpan di /Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json


In [50]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")

# ================= KONFIGURASI =================
WAVEFORM_ROOT = Path('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02')
CATALOG_CSV = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog_CLEAN.csv'
OUTPUT_JSON = "/Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json"

# Parameter MCU-Quake (sesuai paper)
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5

# ================= FUNGSI PREPROCESSING =================

def pick_p_arrival(trace, origin_time, search_window=15):
    """
    Deteksi P-wave otomatis dengan STA/LTA.
    Input/Output: obspy.UTCDateTime
    """
    try:
        # Potong sinyal dari origin hingga search_window ke depan
        tr = trace.copy().trim(origin_time, origin_time + search_window)
        if len(tr.data) < 100:
            return origin_time

        sampling_rate = tr.stats.sampling_rate
        sta_n = int(STA_WIN * sampling_rate)
        lta_n = int(LTA_WIN * sampling_rate)
        
        cft = recursive_sta_lta(tr.data, sta_n, lta_n)
        trigger_indices = np.where(cft > TRIGGER_THRESHOLD)[0]
        
        if len(trigger_indices) > 0:
            pick_idx = trigger_indices[0]
            if pick_idx > int(2 * sampling_rate):
                return tr.stats.starttime + pick_idx / sampling_rate
        
        # Fallback: ambil puncak maksimum sebagai estimasi
        max_idx = np.argmax(np.abs(tr.data))
        if max_idx > 0:
            return tr.stats.starttime + max_idx / sampling_rate
    except:
        pass
    return origin_time


def extract_mcuquake_windows(trace, p_arrival_time):
    """
    Ekstraksi sinyal dan noise PERSIS seperti di Methods paper.
    Input p_arrival_time: obspy.UTCDateTime
    Kembalikan list float panjang 700.
    """
    # 1. Potong Signal (7 detik mulai dari P) dan Noise (7 detik SEBELUM P)
    tr_signal = trace.copy().trim(p_arrival_time, p_arrival_time + SIG_DURATION)
    tr_noise = trace.copy().trim(p_arrival_time - NOISE_DURATION, p_arrival_time)
    
    # 2. Handling data kosong (padding nol)
    for tr in [tr_signal, tr_noise]:
        if len(tr.data) == 0:
            tr.data = np.zeros(int(SAMPLE_RATE * SIG_DURATION))
            tr.stats.sampling_rate = SAMPLE_RATE
            tr.stats.npts = len(tr.data)
    
    # 3. Preprocessing: Detrend
    tr_signal.detrend('simple')
    tr_noise.detrend('simple')
    
    # 4. Resample ke 100 Hz
    if tr_signal.stats.sampling_rate != SAMPLE_RATE:
        tr_signal.resample(SAMPLE_RATE)
    if tr_noise.stats.sampling_rate != SAMPLE_RATE:
        tr_noise.resample(SAMPLE_RATE)
    
    # 5. Normalisasi dengan max absolut dalam 9 detik SETELAH P
    tr_norm = trace.copy().trim(p_arrival_time, p_arrival_time + NORM_WINDOW)
    if len(tr_norm.data) > 0:
        max_val = np.max(np.abs(tr_norm.data))
    else:
        max_val = np.max(np.abs(tr_signal.data))
    
    if max_val == 0:
        max_val = 1.0
        
    signal_data = tr_signal.data / max_val
    noise_data = tr_noise.data / max_val
    
    # 6. Potong/padding agar panjangnya tepat 700 sampel (7 detik * 100 Hz)
    target_len = int(SAMPLE_RATE * SIG_DURATION)
    
    def fix_length(data):
        if len(data) > target_len:
            return data[:target_len]
        elif len(data) < target_len:
            return np.pad(data, (0, target_len - len(data)), 'constant')
        return data
    
    return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()


# ================= FUNGSI UTAMA =================

def build_json_final():
    print("[INFO] Loading katalog BMKG...")
    df = pd.read_csv(CATALOG_CSV)
    
    # KONVERSI KRUSIAL: Semua waktu pakai obspy.UTCDateTime agar kompatibel dengan operasi + float
    df['origin_dt'] = pd.to_datetime(df['origin_time'], utc=True)
    df['origin_utc'] = df['origin_dt'].apply(lambda x: UTCDateTime(x))
    
    has_p_col = 'p_arrival' in df.columns
    if has_p_col:
        df['p_arrival_dt'] = pd.to_datetime(df['p_arrival'], utc=True)
        df['p_arrival_utc'] = df['p_arrival_dt'].apply(lambda x: UTCDateTime(x))
        print("[INFO] Menggunakan kolom P_arrival dari katalog.")
    else:
        print("[INFO] Kolom P_arrival tidak ditemukan. STA/LTA otomatis akan digunakan.")
    
    all_files = list(WAVEFORM_ROOT.rglob("*.mseed"))
    print(f"[INFO] Menemukan {len(all_files)} file waveform.")
    
    output = {}
    count = 0
    failed = 0
    error_logs = []
    
    for file_path in all_files:
        try:
            # ----- 1. Baca header untuk cari rentang waktu file -----
            st_head = read(str(file_path), headonly=True)
            if len(st_head) == 0:
                continue
                
            tr_head = st_head[0]
            file_start = tr_head.stats.starttime  # UTCDateTime
            durasi_file = tr_head.stats.npts / tr_head.stats.sampling_rate
            file_end = file_start + durasi_file   # UTCDateTime + float = OK!
            
            # ----- 2. Cari event di Katalog yang berada dalam rentang file (+/- 60 detik buffer) -----
            # Bandingkan UTCDateTime langsung (bisa dikurangkan)
            mask = (df['origin_utc'] >= file_start - 60) & (df['origin_utc'] <= file_end + 60)
            candidates = df[mask].copy()
            
            if candidates.empty:
                continue
            
            # Ambil event yang origin_time-nya paling dekat dengan START file
            candidates['diff'] = np.abs(candidates['origin_utc'] - file_start)  # pengurangan UTCDateTime menghasilkan float (detik)
            best_idx = candidates['diff'].idxmin()
            best_event = candidates.loc[best_idx]
            
            # ----- 3. Baca data penuh & ambil komponen Z -----
            st_full = read(str(file_path))
            
            try:
                tr_z = st_full.select(component='Z')[0]
            except IndexError:
                # Fallback ke channel pertama jika tidak ada Z
                tr_z = st_full[0]
                if failed < 10:
                    error_logs.append(f"WARNING: No Z in {file_path.name}, using {tr_z.stats.channel}")
            
            # ----- 4. Tentukan P-wave Arrival (semua dalam UTCDateTime) -----
            if has_p_col:
                p_time = best_event['p_arrival_utc']
            else:
                p_time = pick_p_arrival(tr_z, best_event['origin_utc'])
            
            # ----- 5. Ekstraksi sesuai MCU-Quake -----
            sig_list, noise_list = extract_mcuquake_windows(tr_z, p_time)
            
            event_id = str(best_event['Event ID'])
            output[event_id] = {
                "type": "se",
                "Z": sig_list,
                "Z_noise": noise_list,
                "metadata": {
                    "origin_time": str(best_event['origin_dt']),
                    "p_arrival": str(p_time),
                    "mag": float(best_event.get('magnitude', 0)),
                    "depth": float(best_event.get('depth_km', 0)),
                    "file": file_path.name
                }
            }
            
            count += 1
            if count % 100 == 0:
                print(f"[PROGRESS] Berhasil: {count} event, Gagal: {failed}")
        
        except Exception as e:
            failed += 1
            if failed <= 10:
                error_logs.append(f"ERROR on {file_path.name}: {str(e)}")
            continue
    
    # Cetak 10 error pertama untuk debugging
    if error_logs:
        print("\n======= 10 ERROR/DEBUG PERTAMA =======")
        for err in error_logs:
            print(err)
        print("======================================\n")
    
    # Simpan JSON
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)
    
    print(f"\n[SUKSES] Total {count} event tersimpan di {OUTPUT_JSON}")
    print(f"[INFO] Jumlah file gagal total: {failed}")
    print(f"[INFO] Jumlah file tanpa event (di-skip): {len(all_files) - count - failed}")

if __name__ == "__main__":
    build_json_final()

[INFO] Loading katalog BMKG...
[INFO] Kolom P_arrival tidak ditemukan. STA/LTA otomatis akan digunakan.
[INFO] Menemukan 52761 file waveform.
[PROGRESS] Berhasil: 100 event, Gagal: 0
[PROGRESS] Berhasil: 200 event, Gagal: 0
[PROGRESS] Berhasil: 300 event, Gagal: 0
[PROGRESS] Berhasil: 400 event, Gagal: 0
[PROGRESS] Berhasil: 500 event, Gagal: 0
[PROGRESS] Berhasil: 600 event, Gagal: 0
[PROGRESS] Berhasil: 700 event, Gagal: 0
[PROGRESS] Berhasil: 800 event, Gagal: 0
[PROGRESS] Berhasil: 900 event, Gagal: 0
[PROGRESS] Berhasil: 1000 event, Gagal: 0
[PROGRESS] Berhasil: 1100 event, Gagal: 0
[PROGRESS] Berhasil: 1200 event, Gagal: 0
[PROGRESS] Berhasil: 1300 event, Gagal: 0
[PROGRESS] Berhasil: 1400 event, Gagal: 0
[PROGRESS] Berhasil: 1500 event, Gagal: 0
[PROGRESS] Berhasil: 1600 event, Gagal: 0
[PROGRESS] Berhasil: 1700 event, Gagal: 0
[PROGRESS] Berhasil: 1800 event, Gagal: 0
[PROGRESS] Berhasil: 1900 event, Gagal: 0
[PROGRESS] Berhasil: 2000 event, Gagal: 0
[PROGRESS] Berhasil: 2100 e

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json'